# Scale arm: Qwen3-1.7B-Base

Runs the full pipeline on a fourth, larger model to test whether the region
specific component of a concept direction gains a causal effect with scale.

Feng and Steinhardt (ICLR 2024) find binding ID vectors in every *sufficiently
large* model of the Pythia and LLaMA families. The three models in the paper run
124M to 600M and the region specific component is inert in two of them. This arm
asks whether that is a scale effect.

It measures two things:

1. whether the region specific component becomes causally effective
2. whether the dominant region effect in the read out shrinks as the model grows

**Before running:** Runtime, Change runtime type, and pick a **T4 GPU**. On CPU
this takes hours. On a T4 it takes about twenty minutes.

The narratives are committed to the repository, so this runs on byte identical
data to the other three models rather than on its own draw.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'no GPU found')
assert torch.cuda.is_available(), (
    'No GPU. Runtime, Change runtime type, Hardware accelerator: T4 GPU.')
print('torch', torch.__version__, '| cuda', torch.version.cuda)

## 2. Get the code

If the repository is private, this asks for a GitHub token. Create one at
github.com/settings/tokens with `repo` scope. It is read with `getpass`, so it
is not echoed and is not saved into the notebook.

In [ ]:
import os, subprocess, getpass, shutil

REPO = 'Anya-Louni/Makeup-Its-Mind'
if os.path.isdir('/content/repo'):
    shutil.rmtree('/content/repo')

r = subprocess.run(['git','clone','--depth','1',
                    f'https://github.com/{REPO}.git','/content/repo'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('Public clone failed, the repository is probably private.')
    tok = getpass.getpass('GitHub token (repo scope): ')
    r = subprocess.run(['git','clone','--depth','1',
                        f'https://{tok}@github.com/{REPO}.git','/content/repo'],
                       capture_output=True, text=True)
    del tok
    if r.returncode != 0:
        raise SystemExit(r.stderr[-800:])

os.chdir('/content/repo')
print('cloned at', os.getcwd())
print(subprocess.run(['git','log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install -U "transformers>=4.44" "scikit-learn>=1.4" 2>&1 | tail -2
import transformers, sklearn
print('transformers', transformers.__version__, '| sklearn', sklearn.__version__)

## 3. Confirm the narratives match

The pilot set is committed, so the fingerprint below should be identical to the
one printed by the other three models' runs. If it differs, the comparison is
not like for like and the rest of the notebook is not worth running.

In [ ]:
import hashlib, pathlib
for name in ['natural','shuffled']:
    p = pathlib.Path(f'data/pilot/{name}.jsonl')
    h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    n = sum(1 for _ in p.open(encoding='utf-8'))
    print(f'{name:9} {n:5d} narratives   sha256 {h}')

## 4. Extract activations

The pipeline picks up the GPU automatically through `mwm.models.device()`.
Roughly ten minutes on a T4 for both conditions.

In [ ]:
MODEL = 'Qwen/Qwen3-1.7B-Base'
!python -u scripts/03_extract.py --dataset pilot --model $MODEL \
    --conditions natural shuffled 2>&1 | grep -v "it/s\]" | tail -20

## 5. Probe and baselines

In [ ]:
!python -u scripts/04_probe.py --dataset pilot --model $MODEL \
    --layer-stride 3 --skip-mdl --max-train 6000 --max-test 3000 \
    --mlp-hidden 64 2>&1 | tail -30

In [ ]:
!python -u scripts/17_significance.py --dataset pilot --model $MODEL \
    --seeds 1 --max-train 6000 --n-boot 4000 2>&1 | tail -8
!python -u scripts/15_binding_by_distance.py --dataset pilot --model $MODEL \
    --n-boot 1000 2>&1 | tail -4

## 6. The two questions this arm exists to answer

First, does the read out bind at this size, and does the dominant region effect
shrink? Second, does the region specific component of the direction gain a
causal effect?

In [ ]:
!python -u scripts/23_readout_validity.py --dataset pilot --model $MODEL \
    --per-cell 60 2>&1 | tail -18

In [ ]:
!python -u scripts/24_readout_mechanism.py --dataset pilot --model $MODEL \
    --trials 220 2>&1 | tail -18

In [ ]:
!python -u scripts/22_decompose_direction.py --dataset pilot --model $MODEL \
    --entity lips --other-entity eyes --attribute finish --trials 100 2>&1 | tail -18

## 7. Compare against the three smaller models

The committed result files carry the other three, so the scale question can be
read off directly.

In [ ]:
import json, glob

MODELS = [('gpt2-124M','gpt2'), ('pythia-410m','EleutherAI__pythia-410m'),
          ('Qwen3-0.6B','Qwen__Qwen3-0.6B-Base'), ('Qwen3-1.7B','Qwen__Qwen3-1.7B-Base')]

print('REGION SPECIFIC COMPONENT, steered alone, gain g = 2')
print(f"{'model':<13}{'cos':>7}{'diff share':>12}{'shared effect':>15}"
      f"{'region specific difference':>30}{'p':>9}")
for lbl, tag in MODELS:
    f = glob.glob(f'results/decompose_pilot_{tag}_*.json')
    if not f:
        print(f'{lbl:<13} not run'); continue
    d = json.load(open(f[0], encoding='utf-8')); m = d['meta']; s = d['summary']
    sh = s['shared|2.0']['delta_target']['mean']
    df = s['differential|2.0']['difference']
    print(f"{lbl:<13}{m['mean_cos_d1_d2']:>7.3f}"
          f"{100*m['mean_frac_differential']:>11.1f}%{sh:>+15.3f}"
          f"{df['mean']:>+18.3f} [{df['ci95'][0]:+.3f},{df['ci95'][1]:+.3f}]"
          f"{df['p_two_sided']:>9.4f}")

print('\nREAD OUT: does it answer about the region we asked about?')
print(f"{'model':<13}{'read out':<7}{'sensitivity':>13}{'leakage':>11}{'ratio':>8}")
for lbl, tag in MODELS:
    try:
        d = json.load(open(f'results/readout_validity_pilot_{tag}.json', encoding='utf-8'))
    except FileNotFoundError:
        print(f'{lbl:<13} not run'); continue
    for key, nm in (('lips_readout','lips'), ('eyes_readout','eyes')):
        v = d[key]
        print(f"{lbl:<13}{nm:<7}{v['sensitivity']['mean']:>+13.3f}"
              f"{v['leakage']['mean']:>+11.3f}{v['selectivity_ratio']:>8.2f}")

print('\nWHAT PREDICTS THE ANSWER (average over both read outs)')
print(f"{'model':<13}{'lips value':>12}{'eyes value':>12}{'acted last':>12}{'acted more':>12}")
for lbl, tag in MODELS:
    try:
        d = json.load(open(f'results/readout_mechanism_pilot_{tag}.json', encoding='utf-8'))
    except FileNotFoundError:
        print(f'{lbl:<13} not run'); continue
    e = d['effects']
    av = lambda a, b: (e[a]['mean'] + e[b]['mean']) / 2 if a in e and b in e else float('nan')
    print(f"{lbl:<13}{av('lips|v_e1','eyes|v_e1'):>+12.3f}"
          f"{av('lips|v_e2','eyes|v_e2'):>+12.3f}"
          f"{av('lips|v_last','eyes|v_last'):>+12.3f}"
          f"{av('lips|v_more','eyes|v_more'):>+12.3f}")

## 8. Download the results

Only the JSON files. The activations are several GB and are regenerable.

In [ ]:
import shutil, glob, os
os.makedirs('/content/out', exist_ok=True)
got = [f for f in glob.glob('results/*Qwen__Qwen3-1.7B-Base*')
       if f.endswith('.json')]
for f in got:
    shutil.copy(f, '/content/out/')
print(f'{len(got)} result files')
for f in sorted(got):
    print(' ', os.path.basename(f))
shutil.make_archive('/content/qwen17_results', 'zip', '/content/out')
try:
    from google.colab import files
    files.download('/content/qwen17_results.zip')
except Exception as e:
    print('Download it from the file browser on the left:',
          '/content/qwen17_results.zip', f'({e})')

Unzip into `results/` in the local checkout and the comparison tables in the
paper pick the fourth model up automatically.